# Fast Thermal Denoising + Noise-Aware Visible–Thermal Fusion (Final — Progress Loading Fix)

This notebook is designed for Google Colab. It uses CUDA mixed precision when a supported GPU is available and falls back to CPU otherwise. It pairs the MSRS `train/vi` and `train/ir` images by filename, creates reproducible synthetic stripe + Gaussian noise, trains a compact **noise-residual denoiser**, then trains a small noise-aware fusion gate.

### What is different from the earlier versions
- The denoiser predicts the **noise residual**; the output is `noisy thermal − predicted noise`. With zero-initialized residual prediction, the initial denoised image is the noisy input, so it cannot start as a black image.
- The denoiser is supervised against the clean thermal target **and** the known synthetic noise residual.
- Resized grayscale pairs are cached in RAM, loaded with a small thread pool and a live progress bar, so slow Google Drive reads are visible; Drive is read only once instead of on every epoch.
- Training uses 160×160 images, batch size 16, a small CNN, CUDA mixed precision, and no per-batch SSIM/Sobel convolutions.
- Training and validation outputs are checked using noisy-input versus denoised PSNR. The best checkpoint is selected by validation denoising PSNR, not by the combined fusion loss.
- Fusion uses an explicit learned gate that changes the visible/thermal mix according to the estimated noise map. Its target is a **training proxy**, because MSRS does not provide ground-truth fused images.

### Run instructions
Run the cells from top to bottom. Do not load any previous collapsed checkpoints. This notebook saves only `_v5` checkpoint/result files, leaving older files untouched.

**Research note:** synthetic noise performance is not the same as performance on a real noisy thermal camera. Report the synthetic-noise setup and use a separate real-noise test set if available.

### Quick demo (without retraining)
To launch **Thermal Fusion Studio** in Google Colab without downloading the MSRS dataset or running training, run only: (1) the first code cell (`Imports / Configuration`), (2) the `Model Architecture` cell that defines `ThermalNoiseResidualCNN`, `NoiseAwareFusionGate`, and the inference helpers, and (3) the final `THERMAL FUSION STUDIO — INTERACTIVE COLAB APP` cell. Skip dataset discovery, preprocessing, training, and evaluation cells for this demo-only path. The final app cell downloads the public `best_checkpoint_v5.pth` from this repository if it is not found locally or in your own Google Drive.

For full training/reproduction, run the notebook cells in order and provide your own copy of the MSRS dataset in the configured Google Drive folder. Do not use **Run all** for a quick demo because it will execute dataset preparation and training.


In [ ]:
from google.colab import drive  # Drive is mounted only by the dataset/training path below.

import os, glob, time, math, random
from contextlib import nullcontext
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
IMAGE_SIZE = 160
BATCH_SIZE = 16
NUM_WORKERS = 2
EPOCHS = 8
LEARNING_RATE = 1e-3
VAL_FRACTION = 0.10

SEARCH_ROOT = '/content/drive/MyDrive/Research_Project'
CHECKPOINT_DIR = os.path.join(SEARCH_ROOT, 'Checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print('PyTorch:', torch.__version__)
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print(f'Configuration: {IMAGE_SIZE}x{IMAGE_SIZE}, batch={BATCH_SIZE}, epochs={EPOCHS}, mixed precision={USE_AMP}')

In [ ]:
# Full dataset/training path only: mount Drive here, not during the demo-only startup.
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')

VALID_EXT = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
def image_names(folder):
    if not os.path.isdir(folder):
        return set()
    return {os.path.basename(p) for p in glob.glob(os.path.join(folder, '*'))
            if os.path.isfile(p) and os.path.splitext(p)[1].lower() in VALID_EXT}

def count_matching_pairs(train_dir):
    return len(image_names(os.path.join(train_dir, 'vi')) & image_names(os.path.join(train_dir, 'ir')))

train_candidates = []
for current, dirs, files in os.walk(SEARCH_ROOT):
    if 'train' in dirs:
        candidate = os.path.join(current, 'train')
        if os.path.isdir(os.path.join(candidate, 'vi')) and os.path.isdir(os.path.join(candidate, 'ir')):
            count = count_matching_pairs(candidate)
            if count:
                train_candidates.append((count, candidate))

if not train_candidates:
    raise FileNotFoundError(
        f'Could not find MSRS train/vi and train/ir folders under {SEARCH_ROOT}. '
        'Check that Google Drive is mounted and the dataset is extracted.'
    )

pair_count, train_path = max(train_candidates, key=lambda item: item[0])
dataset_root = os.path.dirname(train_path)
visible_dir = os.path.join(train_path, 'vi')
thermal_dir = os.path.join(train_path, 'ir')
test_visible_dir = os.path.join(dataset_root, 'test', 'vi')
test_thermal_dir = os.path.join(dataset_root, 'test', 'ir')
print('Dataset root:', dataset_root)
print('Matching training pairs:', pair_count)
print('Separate test split available:', os.path.isdir(test_visible_dir) and os.path.isdir(test_thermal_dir))
print('Checkpoint directory:', CHECKPOINT_DIR)

In [ ]:
class ThermalNoiseResidualCNN(nn.Module):
    """Predict the added noise, then subtract it from the noisy thermal image."""
    def __init__(self, channels=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, channels, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(channels, 1, 3, padding=1)
        )
        # Start with zero predicted noise => output initially equals the noisy input.
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, noisy):
        return self.net(noisy)


class NoiseAwareFusionGate(nn.Module):
    """Predict a per-pixel visible-image weight from visible, thermal and noise maps."""
    def __init__(self, channels=16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, channels, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(channels, 1, 1)
        )
        # Start with a 60% visible / 40% thermal mix, never a constant black/white image.
        nn.init.zeros_(self.features[-1].weight)
        nn.init.constant_(self.features[-1].bias, math.log(0.60 / 0.40))

    def forward(self, visible, thermal_enhanced, normalized_noise):
        gate = torch.sigmoid(self.features(torch.cat([visible, thermal_enhanced, normalized_noise], dim=1)))
        fused = gate * visible + (1.0 - gate) * thermal_enhanced
        return fused.clamp(0.0, 1.0), gate


def denoise_image(model, noisy):
    predicted_noise = model(noisy)
    denoised = (noisy - predicted_noise).clamp(0.0, 1.0)
    return denoised, predicted_noise


def enhance_thermal_for_fusion(thermal):
    # Per-image contrast normalization for fusion only; denoising targets stay unaltered.
    mean = thermal.mean(dim=(-2, -1), keepdim=True)
    std = thermal.std(dim=(-2, -1), keepdim=True, unbiased=False).clamp_min(0.04)
    return ((thermal - mean) / (3.0 * std) + 0.5).clamp(0.0, 1.0)


def normalize_noise_map(noise_map):
    mean = noise_map.mean(dim=(-2, -1), keepdim=True)
    std = noise_map.std(dim=(-2, -1), keepdim=True, unbiased=False)
    scale = (mean + 2.0 * std).clamp_min(0.02)
    return (noise_map / scale).clamp(0.0, 1.0)


def run_pipeline(denoiser, fusion_net, visible, noisy_thermal):
    denoised, _ = denoise_image(denoiser, noisy_thermal)
    noise_map = (noisy_thermal - denoised).abs()
    noise_norm = normalize_noise_map(noise_map)
    thermal_for_fusion = enhance_thermal_for_fusion(denoised)
    fused, gate = fusion_net(visible, thermal_for_fusion, noise_norm)
    return denoised, noise_map, thermal_for_fusion, fused, gate

model = ThermalNoiseResidualCNN().to(device)
fusion_model = NoiseAwareFusionGate().to(device)
print('Denoiser parameters:', f'{sum(p.numel() for p in model.parameters()):,}')
print('Fusion parameters:', f'{sum(p.numel() for p in fusion_model.parameters()):,}')
print('Both models freshly initialized; no previous checkpoint is loaded.')

In [ ]:
class MSRSThermalDataset(Dataset):
    """Pair grayscale visible/IR images, cache resized uint8 arrays, and add synthetic noise."""
    def __init__(self, visible_dir, thermal_dir, image_size=128, fixed_noise_seed=None, shared_cache=None):
        self.image_size = image_size
        self.fixed_noise_seed = fixed_noise_seed
        if shared_cache is not None:
            self.pairs = shared_cache.pairs
            self.visible_u8 = shared_cache.visible_u8
            self.thermal_u8 = shared_cache.thermal_u8
            return

        visible_files = {os.path.basename(p): p for p in glob.glob(os.path.join(visible_dir, '*'))
                         if os.path.isfile(p) and os.path.splitext(p)[1].lower() in VALID_EXT}
        thermal_files = {os.path.basename(p): p for p in glob.glob(os.path.join(thermal_dir, '*'))
                         if os.path.isfile(p) and os.path.splitext(p)[1].lower() in VALID_EXT}
        common = sorted(set(visible_files) & set(thermal_files))
        if not common:
            raise ValueError(f'No matching visible/thermal files: {visible_dir} and {thermal_dir}')
        self.pairs = [(visible_files[n], thermal_files[n]) for n in common]

        start = time.time()
        print(f'Caching {len(common)} visible/thermal pairs from Drive; progress updates below...', flush=True)

        def read_pair(name):
            vis = cv2.imread(visible_files[name], cv2.IMREAD_GRAYSCALE)
            thm = cv2.imread(thermal_files[name], cv2.IMREAD_GRAYSCALE)
            if vis is None or thm is None:
                raise RuntimeError(f'Cannot read image pair: {visible_files[name]} | {thermal_files[name]}')
            vis = cv2.resize(vis, (image_size, image_size), interpolation=cv2.INTER_AREA)
            thm = cv2.resize(thm, (image_size, image_size), interpolation=cv2.INTER_AREA)
            return vis, thm

        visible_images, thermal_images = [], []
        # Drive file reads are I/O-bound. A small thread pool overlaps file access;
        # tqdm makes slow Drive reads visible instead of appearing frozen.
        with ThreadPoolExecutor(max_workers=6) as pool:
            for vis, thm in tqdm(
                pool.map(read_pair, common),
                total=len(common),
                desc='Reading image pairs',
                unit='pair'
            ):
                visible_images.append(vis)
                thermal_images.append(thm)

        self.visible_u8 = np.stack(visible_images, axis=0)
        self.thermal_u8 = np.stack(thermal_images, axis=0)
        print(f'Cached {len(self.pairs)} resized image pairs in RAM in {time.time()-start:.1f}s.', flush=True)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        vis = self.visible_u8[idx].astype(np.float32) / 255.0
        clean = self.thermal_u8[idx].astype(np.float32) / 255.0
        noisy = clean.copy()
        if self.fixed_noise_seed is None:
            rng = np.random.default_rng()
        else:
            rng = np.random.default_rng(self.fixed_noise_seed + int(idx))

        # Narrow vertical stripes plus Gaussian noise; same corruption is reproducible for validation.
        stripe_offsets = rng.uniform(-0.12, 0.12, size=(self.image_size // 8 + 1,)).astype(np.float32)
        for stripe_index, x in enumerate(range(0, self.image_size, 8)):
            noisy[:, x:min(x + 2, self.image_size)] += stripe_offsets[stripe_index]
        noisy += rng.normal(0.0, 0.05, noisy.shape).astype(np.float32)
        noisy = np.clip(noisy, 0.0, 1.0)

        return (
            torch.from_numpy(vis).unsqueeze(0),
            torch.from_numpy(clean).unsqueeze(0),
            torch.from_numpy(noisy).unsqueeze(0)
        )


# Load from Drive once. The validation view shares the cached arrays and uses fixed noise.
full_train_dataset = MSRSThermalDataset(visible_dir, thermal_dir, IMAGE_SIZE)
validation_dataset = MSRSThermalDataset(
    visible_dir, thermal_dir, IMAGE_SIZE, fixed_noise_seed=20261012, shared_cache=full_train_dataset
)
indices = np.random.default_rng(SEED).permutation(len(full_train_dataset))
val_count = max(1, int(round(len(indices) * VAL_FRACTION)))
val_indices = indices[:val_count].tolist()
train_indices = indices[val_count:].tolist()
train_dataset = Subset(full_train_dataset, train_indices)
val_dataset = Subset(validation_dataset, val_indices)

loader_kwargs = {
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'pin_memory': torch.cuda.is_available(),
    'persistent_workers': NUM_WORKERS > 0
}
train_loader = DataLoader(train_dataset, shuffle=True, drop_last=False, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, drop_last=False, **loader_kwargs)

print(f'All paired images: {len(full_train_dataset)}')
print(f'Train images: {len(train_dataset)} | Validation images: {len(val_dataset)}')
print(f'Batches/epoch: {len(train_loader)} | Validation batches: {len(val_loader)}')
print('Dataset output order: visible, clean thermal, noisy thermal.')

In [ ]:
def psnr_from_mse(mse):
    return 10.0 * math.log10(1.0 / max(float(mse), 1e-12))

@torch.no_grad()
def evaluate_denoiser(denoiser, loader):
    denoiser.eval()
    squared_noisy = 0.0
    squared_denoised = 0.0
    absolute_denoised = 0.0
    pixel_count = 0
    denoised_sum = 0.0
    denoised_sq_sum = 0.0
    for visible, clean, noisy in loader:
        clean = clean.to(device, non_blocking=True)
        noisy = noisy.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            denoised, _ = denoise_image(denoiser, noisy)
        clean_f = clean.float()
        noisy_f = noisy.float()
        denoised_f = denoised.float()
        squared_noisy += F.mse_loss(noisy_f, clean_f, reduction='sum').item()
        squared_denoised += F.mse_loss(denoised_f, clean_f, reduction='sum').item()
        absolute_denoised += F.l1_loss(denoised_f, clean_f, reduction='sum').item()
        pixel_count += clean.numel()
        denoised_sum += denoised_f.sum().item()
        denoised_sq_sum += denoised_f.square().sum().item()
    mse_noisy = squared_noisy / pixel_count
    mse_denoised = squared_denoised / pixel_count
    mean = denoised_sum / pixel_count
    variance = max(0.0, denoised_sq_sum / pixel_count - mean * mean)
    return {
        'noisy_psnr': psnr_from_mse(mse_noisy),
        'denoised_psnr': psnr_from_mse(mse_denoised),
        'denoised_l1': absolute_denoised / pixel_count,
        'denoised_mean': mean,
        'denoised_std': math.sqrt(variance),
        'mse_noisy': mse_noisy,
        'mse_denoised': mse_denoised
    }

# Pre-flight check: the untrained residual model must pass the noisy image through, not output black.
vis0, clean0, noisy0 = next(iter(val_loader))
with torch.no_grad():
    denoised0, _ = denoise_image(model, noisy0.to(device))
print('Initial check — noisy mean/std:', round(noisy0.mean().item(), 4), round(noisy0.std().item(), 4))
print('Initial check — denoised mean/std:', round(denoised0.mean().item(), 4), round(denoised0.std().item(), 4))
if denoised0.std().item() < 0.5 * noisy0.std().item():
    raise RuntimeError('Pre-flight failed: initial denoiser is unexpectedly collapsing.')

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(clean0[0, 0].numpy(), cmap='gray', vmin=0, vmax=1); axes[0].set_title('Clean thermal target')
axes[1].imshow(noisy0[0, 0].numpy(), cmap='gray', vmin=0, vmax=1); axes[1].set_title('Synthetic noisy input')
axes[2].imshow(denoised0[0, 0].cpu().numpy(), cmap='gray', vmin=0, vmax=1); axes[2].set_title('Initial output (identity)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
optimizer_denoise = optim.Adam(model.parameters(), lr=LEARNING_RATE)
optimizer_fusion = optim.Adam(fusion_model.parameters(), lr=LEARNING_RATE)
try:
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_denoise, mode='max', factor=0.5, patience=2, min_lr=1e-4
)

history = {'denoise_loss': [], 'fusion_loss': [], 'val_noisy_psnr': [], 'val_denoised_psnr': []}
best_val_psnr = -float('inf')
best_epoch = 0
best_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_checkpoint_v5.pth')
latest_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'latest_checkpoint_v5.pth')

# Record an initial fallback checkpoint: if learning fails to improve on the noisy input,
# the best model remains the safe identity denoiser instead of a collapsed model.
initial_metrics = evaluate_denoiser(model, val_loader)
print('Initial validation PSNR — noisy:', f"{initial_metrics['noisy_psnr']:.3f} dB",
      '| initial identity denoiser:', f"{initial_metrics['denoised_psnr']:.3f} dB")
best_val_psnr = initial_metrics['denoised_psnr']
initial_checkpoint = {
    'epoch': 0,
    'model_state_dict': model.state_dict(),
    'fusion_model_state_dict': fusion_model.state_dict(),
    'val_psnr': best_val_psnr,
    'history': history,
    'config': {'image_size': IMAGE_SIZE, 'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'seed': SEED}
}
torch.save(initial_checkpoint, best_checkpoint_path)

print(f'Fresh training on {device}: {len(train_dataset)} train / {len(val_dataset)} validation images.')
print('Architecture: noise-residual CNN + lightweight noise-aware fusion gate.')

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    fusion_model.train()
    running_denoise = 0.0
    running_fusion = 0.0
    samples_seen = 0

    for visible, clean, noisy in train_loader:
        visible = visible.to(device, non_blocking=True)
        clean = clean.to(device, non_blocking=True)
        noisy = noisy.to(device, non_blocking=True)
        noise_target = noisy - clean

        # Stage 1: train the denoiser to predict the corruption, not the whole dark image.
        optimizer_denoise.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            predicted_noise = model(noisy)
        denoised = (noisy - predicted_noise).clamp(0.0, 1.0)
        loss_reconstruction = F.l1_loss(denoised.float(), clean.float()) + 0.25 * F.mse_loss(denoised.float(), clean.float())
        loss_noise = F.smooth_l1_loss(predicted_noise.float(), noise_target.float())
        loss_denoise = loss_reconstruction + 0.5 * loss_noise
        if not torch.isfinite(loss_denoise):
            raise RuntimeError('Non-finite denoiser loss detected; training stopped before saving bad weights.')
        scaler.scale(loss_denoise).backward()
        scaler.unscale_(optimizer_denoise)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer_denoise)
        scaler.update()

        # Stage 2: only the fusion gate is updated. Inputs are detached from the denoiser.
        with torch.no_grad():
            denoised_for_fusion = denoised.detach()
            noise_map = (noisy - denoised_for_fusion).abs()
            noise_norm = normalize_noise_map(noise_map)
            thermal_for_fusion = enhance_thermal_for_fusion(denoised_for_fusion)
            gate_target = (0.55 + 0.35 * noise_norm).clamp(0.55, 0.90)
            fusion_target = gate_target * visible + (1.0 - gate_target) * thermal_for_fusion

        optimizer_fusion.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            fused, gate = fusion_model(visible, thermal_for_fusion, noise_norm)
        loss_fusion = F.l1_loss(gate.float(), gate_target.float()) + 0.25 * F.l1_loss(fused.float(), fusion_target.float())
        if not torch.isfinite(loss_fusion):
            raise RuntimeError('Non-finite fusion loss detected; training stopped before saving bad weights.')
        scaler.scale(loss_fusion).backward()
        scaler.unscale_(optimizer_fusion)
        torch.nn.utils.clip_grad_norm_(fusion_model.parameters(), max_norm=1.0)
        scaler.step(optimizer_fusion)
        scaler.update()

        batch_n = visible.shape[0]
        running_denoise += loss_denoise.detach().item() * batch_n
        running_fusion += loss_fusion.detach().item() * batch_n
        samples_seen += batch_n

    train_denoise = running_denoise / samples_seen
    train_fusion = running_fusion / samples_seen
    val_metrics = evaluate_denoiser(model, val_loader)
    scheduler.step(val_metrics['denoised_psnr'])

    history['denoise_loss'].append(train_denoise)
    history['fusion_loss'].append(train_fusion)
    history['val_noisy_psnr'].append(val_metrics['noisy_psnr'])
    history['val_denoised_psnr'].append(val_metrics['denoised_psnr'])

    improved = val_metrics['denoised_psnr'] > best_val_psnr + 1e-4
    if improved:
        best_val_psnr = val_metrics['denoised_psnr']
        best_epoch = epoch

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'fusion_model_state_dict': fusion_model.state_dict(),
        'optimizer_denoise_state_dict': optimizer_denoise.state_dict(),
        'optimizer_fusion_state_dict': optimizer_fusion.state_dict(),
        'val_psnr': val_metrics['denoised_psnr'],
        'best_val_psnr': best_val_psnr,
        'history': history,
        'config': {'image_size': IMAGE_SIZE, 'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'seed': SEED}
    }
    torch.save(checkpoint, latest_checkpoint_path)
    if improved:
        torch.save(checkpoint, best_checkpoint_path)

    print(
        f'Epoch {epoch:02d}/{EPOCHS} | denoise={train_denoise:.5f} | fusion={train_fusion:.5f} '
        f'| val PSNR noisy={val_metrics["noisy_psnr"]:.2f} dB, denoised={val_metrics["denoised_psnr"]:.2f} dB '
        f'| output mean/std={val_metrics["denoised_mean"]:.3f}/{val_metrics["denoised_std"]:.3f} '
        f'| {time.time()-epoch_start:.1f}s'
    )
    if val_metrics['denoised_psnr'] + 0.05 < val_metrics['noisy_psnr']:
        print('WARNING: denoised PSNR is worse than the noisy input; the best checkpoint selector will keep the better model.')

# Load the best validation denoiser and its matching fusion weights.
best = torch.load(best_checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(best['model_state_dict'])
fusion_model.load_state_dict(best['fusion_model_state_dict'])
model.eval(); fusion_model.eval()
print(f'Best checkpoint restored from epoch {best.get("epoch", 0)}; validation denoising PSNR={best["val_psnr"]:.3f} dB')
print('Saved:', best_checkpoint_path)
print('Saved:', latest_checkpoint_path)

In [ ]:
# Final evaluation uses a real test split when present; otherwise the held-out validation subset.
if os.path.isdir(test_visible_dir) and os.path.isdir(test_thermal_dir) and count_matching_pairs(os.path.join(dataset_root, 'test')) > 0:
    final_full_dataset = MSRSThermalDataset(test_visible_dir, test_thermal_dir, IMAGE_SIZE, fixed_noise_seed=20261013)
    final_eval_dataset = final_full_dataset
    evaluation_label = 'separate test split'
else:
    final_eval_dataset = val_dataset
    evaluation_label = 'held-out validation split (no separate test folder found)'

final_loader = DataLoader(
    final_eval_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0
)
final_metrics = evaluate_denoiser(model, final_loader)
print('Evaluation:', evaluation_label)
print('Evaluation images:', len(final_eval_dataset))
print(f'Noisy thermal PSNR:   {final_metrics["noisy_psnr"]:.3f} dB')
print(f'Denoised thermal PSNR:{final_metrics["denoised_psnr"]:.3f} dB')
print(f'PSNR improvement:     {final_metrics["denoised_psnr"]-final_metrics["noisy_psnr"]:+.3f} dB')
print(f'Denoised L1:          {final_metrics["denoised_l1"]:.5f}')
print(f'Denoised mean/std:    {final_metrics["denoised_mean"]:.4f} / {final_metrics["denoised_std"]:.4f}')
if final_metrics['denoised_psnr'] <= final_metrics['noisy_psnr']:
    print('CAUTION: model did not improve PSNR on this evaluation sample set; inspect images before using results in a paper.')

# Correct tuple order: visible, clean thermal, noisy thermal.
vis_t, clean_t, noisy_t = final_eval_dataset[min(0, len(final_eval_dataset)-1)]
vis_t = vis_t.unsqueeze(0).to(device)
clean_t = clean_t.unsqueeze(0).to(device)
noisy_t = noisy_t.unsqueeze(0).to(device)
with torch.no_grad():
    denoised_t, noise_map_t, thermal_enhanced_t, fused_t, gate_t = run_pipeline(model, fusion_model, vis_t, noisy_t)

images = [
    vis_t[0,0].cpu().numpy(),
    noisy_t[0,0].cpu().numpy(),
    clean_t[0,0].cpu().numpy(),
    denoised_t[0,0].cpu().numpy(),
    thermal_enhanced_t[0,0].cpu().numpy(),
    fused_t[0,0].cpu().numpy(),
    noise_map_t[0,0].cpu().numpy()
]
titles = ['Visible input', 'Noisy thermal', 'Clean target', 'Denoised thermal', 'Thermal contrast for fusion', 'Noise-aware fused output', 'Estimated noise map']
cmaps = ['gray', 'inferno', 'inferno', 'inferno', 'gray', 'gray', 'magma']
fig, axes = plt.subplots(1, len(images), figsize=(21, 3.5))
for ax, img, title, cmap in zip(axes, images, titles, cmaps):
    ax.imshow(img, cmap=cmap, vmin=0 if title != 'Estimated noise map' else None, vmax=1 if title != 'Estimated noise map' else None)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
comparison_path = os.path.join(SEARCH_ROOT, 'final_qualitative_comparison_v5.png')
plt.savefig(comparison_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved comparison:', comparison_path)
print('Denoised output min/max:', round(float(denoised_t.min()), 4), '/', round(float(denoised_t.max()), 4))
print('Fusion gate mean:', round(float(gate_t.mean()), 4), '(visible weight; thermal gets the complementary weight)')

In [ ]:
# Lightweight inference benchmark. Report the runtime and image size if using these numbers in a paper.
model.eval(); fusion_model.eval()
dummy_visible = torch.rand(1, 1, IMAGE_SIZE, IMAGE_SIZE, device=device)
dummy_noisy = torch.rand(1, 1, IMAGE_SIZE, IMAGE_SIZE, device=device)
with torch.no_grad():
    for _ in range(5):
        _ = run_pipeline(model, fusion_model, dummy_visible, dummy_noisy)
    if device.type == 'cuda': torch.cuda.synchronize()
    start = time.perf_counter()
    repetitions = 40
    for _ in range(repetitions):
        _ = run_pipeline(model, fusion_model, dummy_visible, dummy_noisy)
    if device.type == 'cuda': torch.cuda.synchronize()
    latency_ms = (time.perf_counter() - start) * 1000.0 / repetitions

parameter_count = sum(p.numel() for p in model.parameters()) + sum(p.numel() for p in fusion_model.parameters())
print(f'Total parameters: {parameter_count:,} ({parameter_count/1000:.2f} K)')
print(f'Input size: {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'Measured model-only latency: {latency_ms:.3f} ms/frame')
print(f'Measured model-only throughput: {1000.0/latency_ms:.1f} FPS')
print('Benchmark excludes file I/O and depends on GPU, warm-up, precision mode, and runtime load.')

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(history['denoise_loss'])+1), history['denoise_loss'], marker='o', label='Denoiser train loss')
plt.plot(range(1, len(history['fusion_loss'])+1), history['fusion_loss'], marker='o', label='Fusion train loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.grid(True); plt.legend(); plt.tight_layout()
loss_plot_path = os.path.join(SEARCH_ROOT, 'training_loss_v5.png')
plt.savefig(loss_plot_path, dpi=200)
plt.show()
plt.figure(figsize=(7,4))
plt.plot(range(1, len(history['val_noisy_psnr'])+1), history['val_noisy_psnr'], marker='o', label='Noisy input PSNR')
plt.plot(range(1, len(history['val_denoised_psnr'])+1), history['val_denoised_psnr'], marker='o', label='Denoised PSNR')
plt.xlabel('Epoch'); plt.ylabel('PSNR (dB)'); plt.grid(True); plt.legend(); plt.tight_layout()
psnr_plot_path = os.path.join(SEARCH_ROOT, 'validation_psnr_v5.png')
plt.savefig(psnr_plot_path, dpi=200)
plt.show()
print('Saved loss plot:', loss_plot_path)
print('Saved validation PSNR plot:', psnr_plot_path)

## Optional: pretrained YOLO demonstration

This last cell is disabled by default so it cannot slow down the training run. Turn `RUN_OPTIONAL_YOLO` to `True` only if you want a qualitative comparison. It uses a pretrained COCO YOLOv8n model; it is **not** a thermal-trained detector and its detections must not be reported as rigorous thermal-object-detection results.

In [ ]:
RUN_OPTIONAL_YOLO = False
if RUN_OPTIONAL_YOLO:
    try:
        from ultralytics import YOLO
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'])
        from ultralytics import YOLO

    detector = YOLO('yolov8n.pt')
    def gray_to_rgb(img):
        return cv2.cvtColor((np.clip(img, 0, 1) * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
    det_images = [gray_to_rgb(images[i]) for i in [0, 1, 5]]
    det_results = [detector.predict(img, verbose=False)[0] for img in det_images]
    rendered = [r.plot()[:, :, ::-1] for r in det_results]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, img, title in zip(axes, rendered, ['Visible', 'Noisy thermal', 'Fused output']):
        ax.imshow(img); ax.set_title(title); ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('YOLO demonstration skipped (RUN_OPTIONAL_YOLO=False). Training and denoising evaluation are complete.')

In [ ]:
# THERMAL FUSION STUDIO — INTERACTIVE COLAB APP

import os
import sys
import json
import time
import tempfile
import subprocess
import importlib.util
from datetime import datetime
from zipfile import ZipFile, ZIP_DEFLATED

import numpy as np
import torch
from PIL import Image, ImageOps

# Install Gradio if needed
if importlib.util.find_spec("gradio") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "gradio>=5,<6"
    ])

import gradio as gr

# Load the trained v5 checkpoint. Prefer an existing local/Drive copy; otherwise fetch it from this public GitHub repository.
from pathlib import Path
from urllib.request import urlretrieve

CHECKPOINT_FILENAME = "best_checkpoint_v5.pth"
CHECKPOINT_URL = "https://raw.githubusercontent.com/jatyasnichedcoding/LightThermNet/main/checkpoints/best_checkpoint_v5.pth"

checkpoint_candidates = [
    Path.cwd() / "checkpoints" / CHECKPOINT_FILENAME,
    Path("/content/drive/MyDrive/Research_Project/Checkpoints") / CHECKPOINT_FILENAME,
    Path("/content") / CHECKPOINT_FILENAME,
]
checkpoint_path = next((p for p in checkpoint_candidates if p.is_file()), None)

if checkpoint_path is None:
    checkpoint_path = Path("/content") / CHECKPOINT_FILENAME
    print("Checkpoint not found locally; downloading the published v5 checkpoint from GitHub...")
    try:
        urlretrieve(CHECKPOINT_URL, checkpoint_path)
    except Exception as exc:
        raise RuntimeError(
            "Could not download the trained checkpoint from GitHub. Verify that "
            "checkpoints/best_checkpoint_v5.pth is committed in the public repository, "
            "or upload the checkpoint to /content/best_checkpoint_v5.pth and rerun this cell."
        ) from exc

if not checkpoint_path.is_file() or checkpoint_path.stat().st_size == 0:
    raise FileNotFoundError(f"Checkpoint file is missing or empty: {checkpoint_path}")

print("Checkpoint file:", checkpoint_path)
checkpoint = torch.load(
    str(checkpoint_path),
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
fusion_model.load_state_dict(checkpoint["fusion_model_state_dict"])

model.to(device).eval()
fusion_model.to(device).eval()
checkpoint_epoch = checkpoint.get("epoch", "unknown")
print("Checkpoint loaded successfully. Epoch:", checkpoint_epoch)
print("Inference device:", device)


def pil_to_tensor(image):
    """Convert an uploaded image to the model's grayscale input format."""
    if image is None:
        raise ValueError("Please upload both a visible and a thermal image.")

    resampling = getattr(Image, "Resampling", Image).LANCZOS
    image = ImageOps.exif_transpose(image).convert("L")
    image = image.resize((IMAGE_SIZE, IMAGE_SIZE), resampling)

    array = np.asarray(image, dtype=np.float32).copy() / 255.0
    tensor = torch.from_numpy(array)[None, None].to(device)

    return tensor, image


def tensor_to_pil(tensor):
    """Convert one grayscale model output into a displayable image."""
    array = tensor.detach().float().cpu().squeeze().numpy()
    array = (np.clip(array, 0.0, 1.0) * 255).round().astype(np.uint8)
    return Image.fromarray(array)


def run_demo(visible_image, thermal_image,
             denoise_strength, thermal_contribution):

    if visible_image is None or thermal_image is None:
        raise gr.Error(
            "Upload both a visible-light image and its matching thermal image."
        )

    visible, visible_pil = pil_to_tensor(visible_image)
    noisy, thermal_pil = pil_to_tensor(thermal_image)

    # Run the trained models
    with torch.inference_mode():
        predicted_noise = model(noisy)

        # Adjust how strongly the estimated noise is removed.
        strength = float(denoise_strength)
        denoised = (noisy - strength * predicted_noise).clamp(0, 1)

        noise_map = (noisy - denoised).abs()
        noise_norm = normalize_noise_map(noise_map)
        thermal_enhanced = enhance_thermal_for_fusion(denoised)

        # Use the trained fusion gate to retain spatially varying weights.
        _, learned_visible_gate = fusion_model(
            visible, thermal_enhanced, noise_norm
        )

        learned_thermal_weight = 1.0 - learned_visible_gate

        # Let the user control the overall thermal contribution while
        # retaining spatial variations learned by the fusion network.
        target_thermal_weight = float(thermal_contribution) / 100.0

        spatial_adjustment = (
            learned_thermal_weight
            - learned_thermal_weight.mean(
                dim=(-2, -1), keepdim=True
            )
        )

        thermal_weight_map = (
            target_thermal_weight + spatial_adjustment
        ).clamp(0, 1)

        fused = (
            (1.0 - thermal_weight_map) * visible
            + thermal_weight_map * thermal_enhanced
        ).clamp(0, 1)

    # Prepare previews
    denoised_pil = tensor_to_pil(denoised)
    enhanced_pil = tensor_to_pil(thermal_enhanced)
    fused_pil = tensor_to_pil(fused)
    noise_pil = tensor_to_pil(noise_norm)
    gate_pil = tensor_to_pil(thermal_weight_map)

    actual_thermal_percent = (
        thermal_weight_map.mean().item() * 100
    )
    learned_thermal_percent = (
        learned_thermal_weight.mean().item() * 100
    )

    # Create a downloadable results package
    session_dir = tempfile.mkdtemp(prefix="thermal_fusion_")

    visible_path = os.path.join(session_dir, "visible_input.png")
    thermal_path = os.path.join(session_dir, "thermal_input.png")
    denoised_path = os.path.join(session_dir, "denoised_thermal.png")
    enhanced_path = os.path.join(session_dir, "enhanced_thermal.png")
    fused_path = os.path.join(session_dir, "fused_output.png")
    noise_path = os.path.join(session_dir, "estimated_noise_map.png")
    gate_path = os.path.join(session_dir, "thermal_weight_map.png")
    metadata_path = os.path.join(session_dir, "metadata.json")

    visible_pil.save(visible_path)
    thermal_pil.save(thermal_path)
    denoised_pil.save(denoised_path)
    enhanced_pil.save(enhanced_path)
    fused_pil.save(fused_path)
    noise_pil.save(noise_path)
    gate_pil.save(gate_path)

    run_info = {
        "timestamp": datetime.now().isoformat(),
        "checkpoint_epoch": checkpoint_epoch,
        "image_size": IMAGE_SIZE,
        "denoising_strength": strength,
        "requested_thermal_contribution_percent": float(
            thermal_contribution
        ),
        "actual_mean_thermal_weight_percent": round(
            actual_thermal_percent, 2
        ),
        "learned_mean_thermal_weight_percent": round(
            learned_thermal_percent, 2
        )
    }

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(run_info, f, indent=2)

    bundle_path = os.path.join(
        session_dir, "thermal_fusion_results.zip"
    )

    with ZipFile(bundle_path, "w", ZIP_DEFLATED) as bundle:
        for path in [
            visible_path, thermal_path, denoised_path,
            enhanced_path, fused_path, noise_path,
            gate_path, metadata_path
        ]:
            bundle.write(path, arcname=os.path.basename(path))

    summary = (
        "Processing complete\n"
        f"Model checkpoint epoch: {checkpoint_epoch}\n"
        f"Input resolution: {IMAGE_SIZE} × {IMAGE_SIZE}\n"
        f"Denoising strength: {strength:.2f}\n"
        f"Requested thermal contribution: {thermal_contribution:.0f}%\n"
        f"Actual mean thermal weight: {actual_thermal_percent:.1f}%\n"
        f"Learned gate's original thermal weight: "
        f"{learned_thermal_percent:.1f}%\n\n"
        "Note: PSNR is not calculated because no clean reference "
        "image was supplied for these uploaded inputs."
    )

    return (
        denoised_pil,
        enhanced_pil,
        fused_pil,
        noise_pil,
        gate_pil,
        summary,
        bundle_path,
        run_info
    )


# Store review feedback in Drive when available.
feedback_root = (
    SEARCH_ROOT if os.path.isdir(SEARCH_ROOT) else "/content"
)
feedback_path = os.path.join(
    feedback_root, "thermal_fusion_feedback.jsonl"
)


def record_feedback(vote, comment, run_info):
    if not run_info:
        return "Process an image pair before submitting feedback."

    record = {
        **run_info,
        "vote": vote,
        "comment": (comment or "").strip()
    }

    with open(feedback_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

    return f"Feedback saved: {vote}. Thank you for reviewing the result."


# Build the interactive application
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # Thermal Fusion Studio
        ### AI-based thermal denoising and visible–thermal fusion

        Upload a **matching pair** of visible-light and thermal images.
        Process them, inspect the results, adjust the settings, and redo.
        """
    )

    run_state = gr.State(None)

    with gr.Row():
        visible_input = gr.Image(
            label="1. Visible-light image",
            type="pil"
        )
        thermal_input = gr.Image(
            label="2. Thermal image",
            type="pil"
        )

    with gr.Row():
        denoise_slider = gr.Slider(
            minimum=0.0,
            maximum=1.5,
            value=1.0,
            step=0.05,
            label="Denoising strength"
        )

        thermal_slider = gr.Slider(
            minimum=0,
            maximum=100,
            value=40,
            step=5,
            label="Thermal contribution (%)"
        )

    with gr.Row():
        process_button = gr.Button(
            "Denoise + Fuse",
            variant="primary"
        )
        redo_button = gr.Button("Redo with current settings")

    gr.Markdown("## Results")

    with gr.Row():
        denoised_output = gr.Image(
            label="3. Denoised thermal"
        )
        enhanced_output = gr.Image(
            label="4. Contrast-enhanced thermal"
        )

    with gr.Row():
        fused_output = gr.Image(
            label="5. Fused output"
        )
        noise_output = gr.Image(
            label="6. Estimated noise map"
        )
        gate_output = gr.Image(
            label="7. Thermal weight map"
        )

    summary_output = gr.Textbox(
        label="Processing details",
        lines=6,
        interactive=False
    )

    download_output = gr.File(
        label="Download all results (.zip)"
    )

    gr.Markdown("## Review this result")

    feedback_comment = gr.Textbox(
        label="Comments (optional)",
        placeholder="What do you like or want improved?",
        lines=2
    )

    with gr.Row():
        like_button = gr.Button("👍 I like it")
        dislike_button = gr.Button("👎 Not satisfied")

    feedback_status = gr.Textbox(
        label="Feedback status",
        interactive=False
    )

    outputs = [
        denoised_output,
        enhanced_output,
        fused_output,
        noise_output,
        gate_output,
        summary_output,
        download_output,
        run_state
    ]

    inputs = [
        visible_input,
        thermal_input,
        denoise_slider,
        thermal_slider
    ]

    process_button.click(
        fn=run_demo,
        inputs=inputs,
        outputs=outputs
    )

    redo_button.click(
        fn=run_demo,
        inputs=inputs,
        outputs=outputs
    )

    like_button.click(
        fn=lambda comment, state: record_feedback(
            "like", comment, state
        ),
        inputs=[feedback_comment, run_state],
        outputs=feedback_status
    )

    dislike_button.click(
        fn=lambda comment, state: record_feedback(
            "not_satisfied", comment, state
        ),
        inputs=[feedback_comment, run_state],
        outputs=feedback_status
    )

print("Thermal Fusion Studio is ready.")
print("A temporary demo link will appear below.")
demo.queue().launch(share=True, debug=False)